# pandas 03. 集約・重複・並べ替え

`groupby` の引数と、「キーごとに1行だけ選ぶ」の書き方をやります。

進み方はこれまでと同じで、**A と B を見比べる → 答え合わせ → ミニ練習1問**です。

このノートの 3章 は、あとの quest でつまずきやすいところなので、
ミニ練習を少し多めに置いてあります。

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

NULLISH = {"NULL", "N/A", "-", ""}

# orders.csv を、型を整えた状態で読む (pandas-01 でやったこと)
def load():
    df = pd.read_csv("/data/orders.csv", dtype=str, keep_default_na=False)
    df = df.map(lambda s: None if str(s).strip() in NULLISH else s)
    df["qty"] = pd.to_numeric(df["qty"]).astype("Int64")
    df["amount"] = pd.to_numeric(df["amount"].str.replace(",", "", regex=False)).astype("Int64")
    df["region"] = df["region"].str.lower()
    return df

df = load()
df

---
## 1. groupby の基本

### 1-1. as_index

グループのキーを index に入れるか、ふつうの列として残すかの違いです。

In [ ]:
# A: as_index=True (既定)
a = df.groupby("region")["amount"].sum()
print(type(a).__name__)
a

In [ ]:
# B: as_index=False
b = df.groupby("region", as_index=False)["amount"].sum()
print(type(b).__name__)
b

<details>
<summary>答え合わせ</summary>

- `A` は **Series** で、`region` が index になります
- `B` は **DataFrame** で、`region` が列として残ります

index に入ると `df["region"]` で触れなくなり、あとの `merge` や書き出しで少し面倒です。
**結果を次の処理に渡すなら `as_index=False`** にしておくと楽です。

`A` から `B` にしたいときは `.reset_index()` でも同じことができます。

</details>

In [ ]:
# ミニ練習: status ごとの amount 合計を、status を列として残したまま出す

ans = ...   # ここに書く

assert list(ans.columns) == ["status", "amount"], list(ans.columns)
assert len(ans) == 3
print("OK")

<details>
<summary>答え</summary>

```python
ans = df.groupby("status", as_index=False)["amount"].sum()
```

</details>

### 1-2. dropna

グループのキーが欠損している行はどうなるか、という話です。

In [ ]:
# A: dropna=True (既定)
a = df.groupby("customer_id", as_index=False)["amount"].sum()
print("行数:", len(a), " 合計:", a["amount"].sum())
a

In [ ]:
# B: dropna=False
b = df.groupby("customer_id", as_index=False, dropna=False)["amount"].sum()
print("行数:", len(b), " 合計:", b["amount"].sum())
b

<details>
<summary>答え合わせ</summary>

**既定では、キーが欠損している行は黙って消えます。**
`O-010` の `customer_id` が欠損しているので、`A` にはその 1960 円が入っていません。

エラーは出ないので、集約の前後で合計を見比べる癖をつけると気づけます。

```python
print(df["amount"].sum(), a["amount"].sum())
```

ちなみに SQL の `GROUP BY` は NULL も1グループとして扱います。
SQL から pandas に移すときに数字が変わる原因の1つです。

</details>

In [ ]:
# ミニ練習: customer_id が欠損の行も1グループとして数える (行数を出す)

ans = ...   # ここに書く

assert len(ans) == 6, f"6行のはず: {len(ans)}"
assert ans["amount"].sum() == 23220
print("OK")

<details>
<summary>答え</summary>

```python
ans = df.groupby("customer_id", as_index=False, dropna=False)["amount"].sum()
```

</details>

### 1-3. size と count

「件数」の数え方が2つあります。欠損の扱いが違います。

In [ ]:
# A: size
a = df.groupby("region").size()
print(a)
print("合計:", a.sum())

In [ ]:
# B: count
b = df.groupby("region")["amount"].count()
print(b)
print("合計:", b.sum())

<details>
<summary>答え合わせ</summary>

- `size` は**行数**。欠損も数えます
- `count` は**欠損でない数**。`O-006` の `amount` が欠損なので north が1少なくなります

SQL の `COUNT(*)` が `size`、`COUNT(col)` が `count` に当たります。

「件数」と言われたら、どちらのつもりかを確認しておくと安全です。
注文の件数なら `size`、金額が入っている注文の件数なら `count` です。

</details>

In [ ]:
# ミニ練習: status ごとの行数を出す

ans = ...   # ここに書く

assert ans["completed"] == 9, ans.to_dict()
assert ans.sum() == 12
print("OK")

<details>
<summary>答え</summary>

```python
ans = df.groupby("status").size()
```

</details>

### 1-4. agg の書き方

複数の集約をまとめる書き方が2つあります。

In [ ]:
# A: 辞書で指定
a = df.groupby("region", as_index=False).agg({"amount": "sum", "qty": "max"})
a

In [ ]:
# B: 名前付きで指定
b = df.groupby("region", as_index=False).agg(
    total_amount=("amount", "sum"),
    max_qty=("qty", "max"),
    n_orders=("order_id", "size"),
)
b

<details>
<summary>答え合わせ</summary>

`B`(named aggregation)のほうがおすすめです。理由は2つあります。

- **出力の列名を自分で決められます。** `A` は元の列名のままです
- **同じ列に複数の集約をかけられます**

```python
df.groupby("region").agg(
    total=("amount", "sum"),
    avg=("amount", "mean"),      # 同じ列に2つ
)
```

集約関数は文字列(`"sum"`)でも自作の関数でも渡せます。
文字列で足りるなら文字列のほうが速いです。

</details>

In [ ]:
# ミニ練習: region ごとに amount の合計 total と平均 avg を出す (名前付きで)

ans = ...   # ここに書く

assert list(ans.columns) == ["region", "total", "avg"], list(ans.columns)
assert ans["total"].sum() == 23220
print("OK")

<details>
<summary>答え</summary>

```python
ans = df.groupby("region", as_index=False).agg(
    total=("amount", "sum"),
    avg=("amount", "mean"),
)
```

</details>

---
## 2. transform — 集約した値を元の行に戻す

### 2-1. agg と transform

「地域ごとの合計」が欲しいとき、集計表として欲しいのか、
元の行に付けたいのかで使うものが変わります。

In [ ]:
# A: agg (行が減る)
a = df.groupby("region")["amount"].sum()
print("行数:", len(a))
a

In [ ]:
# B: transform (行が減らない)
b = df.assign(region_total=df.groupby("region")["amount"].transform("sum"))
print("行数:", len(b))
b[["order_id", "region", "amount", "region_total"]]

<details>
<summary>答え合わせ</summary>

`transform` は**元と同じ行数**の結果を返し、各行にそのグループの値を配ります。

比率や偏差を出すときに便利です。

```python
df["share"] = df["amount"] / df.groupby("region")["amount"].transform("sum")
```

`agg` してから `merge` で戻す、と同じことを1行でやっています。
`merge` を書きたくなったら、まず `transform` で済まないか考えてみてください。

</details>

In [ ]:
# ミニ練習: 各行の amount が、その region の合計に占める割合を share 列にする

ans = ...   # ここに書く

assert "share" in ans.columns
assert round(ans.loc[0, "share"], 3) == 0.293, ans.loc[0, "share"]
print("OK")

<details>
<summary>答え</summary>

```python
ans = df.assign(
    share=df["amount"] / df.groupby("region")["amount"].transform("sum")
)
```

</details>

---
## 3. キーごとに1行だけ選ぶ

ここがこのノートの本題です。「同じ order_id が複数あるときに最新の1行だけ残す」
という処理は、書き方によって**結果が変わります**。

まず、そういう状況のデータを作ります。

In [ ]:
# O-001 の訂正が後から届いた想定。
# 訂正のほうが新しい (ingested_at が後) が、amount が欠損している。
dup = pd.DataFrame([
    {"order_id": "O-001", "ingested_at": "2024-02-01 10:00", "amount": 2400, "status": "completed"},
    {"order_id": "O-001", "ingested_at": "2024-02-02 10:00", "amount": None,  "status": "cancelled"},
    {"order_id": "O-002", "ingested_at": "2024-02-01 10:00", "amount": 980,   "status": "completed"},
])
dup["amount"] = dup["amount"].astype("Int64")
dup

### 3-1. groupby().last() は「最後の行」ではない

「order_id ごとに ingested_at が最新の行」を取りたいときの、2つの書き方です。

In [ ]:
# A: groupby().last()
a = dup.sort_values("ingested_at").groupby("order_id", as_index=False).last()
a

In [ ]:
# B: drop_duplicates(keep='last')
b = dup.sort_values("ingested_at").drop_duplicates(subset="order_id", keep="last")
b

<details>
<summary>答え合わせ</summary>

**O-001 の行を見比べてください。**

`A` の `amount` は **2400** です。最新の行の `amount` は欠損なのに、値が入っています。

`GroupBy.last()` は「グループの最後の行」ではなく、
**列ごとに、最後の欠損でない値**を返すからです。結果として

- `status` は最新行の `cancelled`
- `amount` は最新行が欠損なので、**1つ前の行の 2400**

という、**どの行にも存在しない行**ができあがります。

`B` は行そのものを1本選ぶので、`amount` は欠損のまま残ります。こちらが期待どおりです。

`first()` も同じ性質を持っています。

</details>

In [ ]:
# ミニ練習: dup から「order_id ごとに ingested_at が最も古い1行」を取る
#           (欠損は欠損のまま残ること)

ans = ...   # ここに書く

assert len(ans) == 2
assert ans["status"].tolist() == ["completed", "completed"], ans["status"].tolist()
print("OK")

<details>
<summary>答え</summary>

```python
ans = (dup.sort_values("ingested_at")
          .drop_duplicates(subset="order_id", keep="first"))
```

</details>

### 3-2. head(1) / tail(1) / nth(0)

「行を選ぶ」ほうの書き方です。こちらは存在しない行を作りません。

In [ ]:
# A: groupby().tail(1)
a = dup.sort_values("ingested_at").groupby("order_id").tail(1)
a

In [ ]:
# B: groupby().nth(-1)
b = dup.sort_values("ingested_at").groupby("order_id").nth(-1)
b

<details>
<summary>答え合わせ</summary>

どちらも**行を選ぶ**ので、欠損はそのまま残ります。`drop_duplicates` と同じ結果です。

| 書き方 | 何を返すか | 存在しない行ができるか |
| --- | --- | --- |
| `.last()` / `.first()` | **列ごとの非null値** | **できる** |
| `.tail(1)` / `.head(1)` | 行 | できない |
| `.nth(0)` / `.nth(-1)` | 行 | できない |
| `drop_duplicates(keep=...)` | 行 | できない |

`tail(1)` は元の index を保つので、`reset_index(drop=True)` を足すことが多いです。

「1行選ぶ」つもりのときは `.last()` を使わない、とだけ覚えておけば大丈夫です。

</details>

In [ ]:
# ミニ練習: tail(1) を使って「order_id ごとに最新の1行」を取り、index を振り直す

ans = ...   # ここに書く

assert ans.index.tolist() == [0, 1]
assert pd.isna(ans.set_index("order_id").loc["O-001", "amount"]), "O-001 の amount は欠損のまま"
print("OK")

<details>
<summary>答え</summary>

```python
ans = (dup.sort_values("ingested_at")
          .groupby("order_id")
          .tail(1)
          .reset_index(drop=True))
```

</details>

### 3-3. 順序が決まらないとどうなるか

`ingested_at` が同じ値だったら、どちらの行が選ばれるでしょうか。

In [ ]:
# A: 同着のまま選ぶ
tie = pd.DataFrame([
    {"order_id": "O-001", "ingested_at": "2024-02-01 10:00", "amount": 100},
    {"order_id": "O-001", "ingested_at": "2024-02-01 10:00", "amount": 999},
])
print(tie.sort_values("ingested_at").drop_duplicates("order_id", keep="last"))

In [ ]:
# B: 2つめのキーを足す
tie = pd.DataFrame([
    {"order_id": "O-001", "ingested_at": "2024-02-01 10:00", "amount": 100},
    {"order_id": "O-001", "ingested_at": "2024-02-01 10:00", "amount": 999},
])
print(tie.sort_values(["ingested_at", "amount"]).drop_duplicates("order_id", keep="last"))

<details>
<summary>答え合わせ</summary>

`A` は「たまたま後ろにあったほう」が選ばれます。いまの環境では元の順が保たれますが、
**保証されているわけではありません**。

入力ファイルの順番が変わったり、読む順番が変わったりすると結果が変わります。
流すたびに数字が変わる、といういちばん困る壊れ方です。

`B` のように**2つめのキーを足して、並び順を一意に決めて**おきます。

</details>

In [ ]:
# ミニ練習: ingested_at が同じときは amount の小さいほうを残す

ans = ...   # ここに書く

assert ans["amount"].tolist() == [100], ans["amount"].tolist()
print("OK")

<details>
<summary>答え</summary>

```python
ans = (tie.sort_values(["ingested_at", "amount"])
          .drop_duplicates("order_id", keep="first"))
```

</details>

---
## 4. 重複を調べる

### 4-1. duplicated の keep

どれを「重複」と見なすかを選べます。

In [ ]:
# A: keep='first' (既定)
s = pd.Series(["a", "b", "a", "c", "a"])
print(s.duplicated().tolist())
print("重複とされた数:", s.duplicated().sum())

In [ ]:
# B: keep=False
s = pd.Series(["a", "b", "a", "c", "a"])
print(s.duplicated(keep=False).tolist())
print("重複とされた数:", s.duplicated(keep=False).sum())

<details>
<summary>答え合わせ</summary>

- `keep="first"` は**2回目以降**を True にします(最初は残す)
- `keep="last"` は**最後以外**を True
- `keep=False` は**重複しているものを全部** True

中身を目で確かめたいときは `keep=False` です。
既定のままだと最初の1件が見えないので、比べようがありません。

```python
df[df.duplicated("order_id", keep=False)].sort_values("order_id")
```

これが「重複を調べる」ときの定番の形です。

</details>

In [ ]:
# ミニ練習: dup のうち、order_id が重複している行を「全部」取り出す

ans = ...   # ここに書く

assert len(ans) == 2, f"2行のはず: {len(ans)}"
assert set(ans["order_id"]) == {"O-001"}
print("OK")

<details>
<summary>答え</summary>

```python
ans = dup[dup.duplicated("order_id", keep=False)]
```

</details>

### 4-2. subset を指定するかどうか

「行全体が同じ」を重複とするか、「キーが同じ」を重複とするかの違いです。

In [ ]:
# A: subset なし
a = dup.drop_duplicates()
print("行数:", len(a))
a

In [ ]:
# B: subset='order_id'
b = dup.drop_duplicates(subset="order_id")
print("行数:", len(b))
b

<details>
<summary>答え合わせ</summary>

- `A` は**全部の列が一致する行**だけを重複と見なします。
  同じデータが二重に届いたときに効きます。`dup` には完全一致が無いので減りません
- `B` は `order_id` が同じなら重複と見なします。訂正を1本にまとめたいときはこちらです

両方必要になることもよくあります(完全重複を落としてから、キーで最新を選ぶ)。

</details>

In [ ]:
# ミニ練習: df から region と status の組み合わせの種類を取り出す

ans = ...   # ここに書く

assert list(ans.columns) == ["region", "status"]
assert len(ans) == 7, f"7行のはず: {len(ans)}"
print("OK")

<details>
<summary>答え</summary>

```python
ans = df[["region", "status"]].drop_duplicates()
```

</details>

---
## 5. 並べ替え

### 5-1. na_position

欠損はどこに並ぶでしょうか。

In [ ]:
# A: 既定 (na_position='last')
a = df.sort_values("amount")
a[["order_id", "amount"]]

In [ ]:
# B: na_position='first'
b = df.sort_values("amount", na_position="first")
b[["order_id", "amount"]]

<details>
<summary>答え合わせ</summary>

**昇順でも降順でも、既定では欠損が最後に来ます。**
`ascending=False` にしても欠損は末尾のままです。

なので `sort_values(ascending=False).head(1)` で「最大の行」を取ると、
ちゃんと最大の行が取れます(欠損の行ではありません)。
`na_position="first"` にすると逆になるので、そこだけ覚えておきます。

複数列で向きを変えたいときはリストで渡します。

```python
df.sort_values(["region", "amount"], ascending=[True, False])
```

</details>

In [ ]:
# ミニ練習: amount の昇順で並べ、欠損を先頭に持ってくる。先頭の order_id は?

ans = ...   # ここに書く

assert ans == "O-006", f"O-006 のはず: {ans}"
print("OK")

<details>
<summary>答え</summary>

```python
ans = df.sort_values("amount", na_position="first")["order_id"].iloc[0]
```

</details>

### 5-2. rank の method

同じ値が並んだときの順位の付け方です。
SQL の `RANK` / `DENSE_RANK` / `ROW_NUMBER` に対応します。

In [ ]:
# A: min と dense
s = pd.Series([10, 20, 20, 30])
print("min   :", s.rank(method="min").tolist())
print("dense :", s.rank(method="dense").tolist())

In [ ]:
# B: first と average
s = pd.Series([10, 20, 20, 30])
print("first   :", s.rank(method="first").tolist())
print("average :", s.rank(method="average").tolist())

<details>
<summary>答え合わせ</summary>

| method | 同じ値の扱い | SQL |
| --- | --- | --- |
| `min` | 同じ順位。次は飛ぶ (1,2,2,4) | `RANK()` |
| `dense` | 同じ順位。次は飛ばない (1,2,2,3) | `DENSE_RANK()` |
| `first` | 出てきた順に別々の順位 (1,2,3,4) | `ROW_NUMBER()` |
| `average` | **既定**。平均 (1,2.5,2.5,4) | — |

既定が `average` なので、何も指定しないと小数が出ます。
「順位」が欲しいときは、たいてい `min` か `dense` を指定します。

</details>

In [ ]:
# ミニ練習: 同じ値には同じ順位を付け、次の順位を飛ばさない付け方で順位を出す

s = pd.Series([10, 20, 20, 30])

ans = ...   # ここに書く

assert ans.tolist() == [1.0, 2.0, 2.0, 3.0], ans.tolist()
print("OK")

<details>
<summary>答え</summary>

```python
ans = s.rank(method="dense")
```

</details>

---
## 仕上げ

In [ ]:
# 仕上げ1: region ごとに、注文件数・金額合計・金額平均を出す。
#          列名は n_orders / total / avg。region は列として残す。

ans = ...   # ここに書く

assert list(ans.columns) == ["region", "n_orders", "total", "avg"], list(ans.columns)
assert len(ans) == 4
assert ans.set_index("region").loc["east", "n_orders"] == 4
assert ans["total"].sum() == 23220, f"合計が合わない: {ans['total'].sum()}"
print("OK")

<details>
<summary>答え</summary>

```python
ans = df.groupby("region", as_index=False).agg(
    n_orders=("order_id", "size"),
    total=("amount", "sum"),
    avg=("amount", "mean"),
)
```

</details>

In [ ]:
# 仕上げ2: 下の raw から「order_id ごとに ingested_at が最新の1行」を取り出す。
#          存在しない行を作らないこと。order_id 昇順、index は振り直す。

raw = pd.DataFrame([
    {"order_id": "O-002", "ingested_at": "2024-02-01 09:00", "amount": 980,  "status": "pending"},
    {"order_id": "O-001", "ingested_at": "2024-02-01 10:00", "amount": 2400, "status": "completed"},
    {"order_id": "O-001", "ingested_at": "2024-02-02 10:00", "amount": None, "status": "cancelled"},
    {"order_id": "O-003", "ingested_at": "2024-02-01 11:00", "amount": 3600, "status": "completed"},
])
raw["amount"] = raw["amount"].astype("Int64")

ans = ...   # ここに書く

assert len(ans) == 3, f"3行のはず: {len(ans)}"
assert ans["order_id"].tolist() == ["O-001", "O-002", "O-003"]
assert pd.isna(ans.loc[0, "amount"]), "O-001 の amount は欠損のまま残るはず"
assert ans.loc[0, "status"] == "cancelled"
print("OK")

<details>
<summary>答え</summary>

```python
ans = (raw.sort_values(["ingested_at", "order_id"])
          .drop_duplicates(subset="order_id", keep="last")
          .sort_values("order_id")
          .reset_index(drop=True))
```

</details>

In [ ]:
# 仕上げ3: 仕上げ2の ans から、amount が欠損している行を除く。
#          (この順番が大事。先に除くと O-001 の古い行が生き残ってしまう)

ans3 = ...   # ここに書く

assert ans3["order_id"].tolist() == ["O-002", "O-003"], ans3["order_id"].tolist()
print("OK")

<details>
<summary>答え</summary>

```python
ans3 = ans[ans["amount"].notna()]
```

</details>

---
## まとめ

| 書き方 | 意味 | 注意 |
| --- | --- | --- |
| `groupby(as_index=False)` | キーを列として残す | 次の処理に渡すならこちら |
| `groupby(dropna=False)` | キーが欠損の群も残す | **既定は消える。合計が変わる** |
| `size` / `count` | 行数 / 欠損でない数 | `COUNT(*)` と `COUNT(col)` |
| `agg(name=(col, fn))` | 名前付き集約 | 列名を自分で決められる |
| `transform` | 集約値を元の行に配る | 行数が変わらない |
| **`.last()` / `.first()`** | **列ごとの非null値** | **存在しない行ができる** |
| `.tail(1)` / `.nth(-1)` | 行を選ぶ | 安全 |
| `drop_duplicates(subset, keep)` | 行を選ぶ | 安全。第一候補 |
| `duplicated(keep=False)` | 重複を全部 True | 調べるとき |
| `sort_values(na_position=)` | 欠損の位置 | 既定は末尾 |
| `rank(method=)` | 同じ値の順位 | 既定は average で小数が出る |

### 「キーごとに最新の1行」の定型

```python
(df.sort_values(["ingested_at", "order_id"])          # 並び順を一意にする
   .drop_duplicates(subset="order_id", keep="last")   # 行を選ぶ
   .reset_index(drop=True))
```

**選んでから、捨てる。** いらない行を除くのは、このあとです。

次は `sql-01-null-join-window.ipynb` です。